# 1st Round

In [0]:
from pyspark.sql.functions import col, to_date, expr, avg, rank, dense_rank
from pyspark.sql.window import Window

# Create df1
data1 = [
    (1, 34123, 27, 6700, "2017-01-25", "202401"),
    (2, 34123, 27, 2000, "2017-01-25", "202401"),
    (1, 34124, 54, 6600, "2018-02-21", "202405"),
    (3, 34128, 45, 8000, "2021-09-02", "202203"),
    (3, 35128, 40, 7200, "2021-09-02", "202203"),
    (3, 36145, 42, 7700, "2019-01-25", "202401"),
    (2, 35128, 40, 720, "2018-04-25", "202309"),
    (1, 30003, 29, 700, "2020-09-25", "202312"),
    (1, 30104, 25, 6100, "2013-12-25", "202401")
]
columns1 = ["product_id", "cust_id", "cust_age", "out_bal", "orig_date", "write_off_date"]
df1 = spark.createDataFrame(data1, columns1)

# Create df2
data2 = [
    (1, "MTG"),
    (2, "PL"),
    (3, "Card")
]
columns2 = ["product_id", "product_name"]
df2 = spark.createDataFrame(data2, columns2)
df1.show()
df2.show()

In [0]:
# 1. Difference between orig_date and write_off_date (in months)
df1 = df1.withColumn("orig_date_dt", to_date(col("orig_date"), "yyyy-MM-dd")) \
         .withColumn("write_off_date_dt", expr("to_date(concat(substr(write_off_date,1,4),'-',substr(write_off_date,5,2),'-01'), 'yyyy-MM-dd')")) \
         .withColumn("month_diff", expr("months_between(write_off_date_dt, orig_date_dt)"))
display(df1.select("product_id", "cust_id", "orig_date", "write_off_date","write_off_date_dt", "month_diff"))

In [0]:
# 2. Product name wise average cust_age
df_join = df1.join(df2, "product_id")
avg_age = df_join.groupBy("product_name").agg(avg("cust_age").alias("avg_cust_age"))
display(avg_age)


In [0]:
# 3. Product name, cust_id, 2nd highest outstanding balance
window_spec = Window.partitionBy("cust_id").orderBy(col("out_bal").desc())
df_ranked = df_join.withColumn("rank", dense_rank().over(window_spec))
second_highest = df_ranked.filter(col("rank") == 2).select("product_name", "cust_id", "out_bal")
display(second_highest)

In [0]:
# 4. Rank and Dense Rank example (product_name, cust_id, out_bal)
df_ranked = df_join.withColumn("rank", rank().over(window_spec)) \
                   .withColumn("dense_rank", dense_rank().over(window_spec))
display(df_ranked.select("product_name", "cust_id", "out_bal", "rank", "dense_rank"))

Let’s break this down step by step:

### 📊 Tables

**Table A**
```
1
1
2
3
```

**Table B**
```
1
2
2
```

---

### 🔗 Inner Join (A ⋈ B)
Inner join returns only the matching values between A and B.

- Value `1`:  
  - Table A has **2 occurrences**  
  - Table B has **1 occurrence**  
  → Matches = 2 × 1 = **2 records**

- Value `2`:  
  - Table A has **1 occurrence**  
  - Table B has **2 occurrences**  
  → Matches = 1 × 2 = **2 records**

- Value `3`:  
  - Table A has **1 occurrence**  
  - Table B has **0 occurrences**  
  → Matches = 0

✅ **Total Inner Join Records = 2 + 2 = 4**

---

### ⬅️ Left Join (A ⟕ B)
Left join keeps all rows from Table A, and matches from Table B if available.

- Value `1`:  
  - A has 2 rows → each matches with B’s 1 row → **2 records**

- Value `2`:  
  - A has 1 row → matches with B’s 2 rows → **2 records**

- Value `3`:  
  - A has 1 row → no match in B → **1 record (with NULLs for B)**

✅ **Total Left Join Records = 2 + 2 + 1 = 5**

---

### ✨ Final Answer
- **Inner Join → 4 records**  
- **Left Join → 5 records**

---


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

data = [("Jan",800),("Feb",80),("Mar",100),("Apr",200),("May",50),("Jun",100)]
df = spark.createDataFrame(data, ["Month","Amount"])

windowSpec = Window.orderBy("Month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = df.withColumn("Cumulative_Sum", F.sum("Amount").over(windowSpec))
df.show()


# 2nd Round

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

data = [
    (1, "AR", "a", "2025-06-22", 200),
    (1, "AR", "d", "2025-08-22", 500),
    (1, "AR", "a", "2025-09-22", None),
    (1, "AR", "b", "2025-07-15", 300),
    (1, "AR", "c", "2025-08-10", 400),
    (1, "AR", "b", "2025-09-05", 500),
    (1, "AR", "a", "2025-05-20", 150),
    (1, "AR", "d", "2025-06-18", 250),
    (1, "AR", "c", "2025-07-22", None)
]

schema = StructType([
    StructField("cust_id", IntegerType(), True),
    StructField("cust_name", StringType(), True),
    StructField("DQ_grade", StringType(), True),
    StructField("date", StringType(), True),
    StructField("balance", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)
df = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# replace the blank values of balance column with the average of previous two values?
new_df = df.withColumn("prev_avg",avg("balance").over(Window.partitionBy("cust_id","cust_name").orderBy("date").rowsBetween(-2,-1))).withColumn("balance",when(col("balance").isNull(),col("prev_avg")).otherwise(col("balance"))).drop("prev_avg")
display(new_df)

In [0]:
# max time spend in which grade?
final_df = new_df.groupBy("DQ_grade").agg(count("date").alias("cnt")).limit(1)
display(final_df)

In [0]:
# What would be the output of the following?

print(0.4 + 0.2 == 0.6)

In [0]:
# SCD2 implementation using pyspark


from pyspark.sql.functions import col, lit, lag, lead, current_date, when
from pyspark.sql.window import Window

# Sample SCD2 source data
source_data = [
    (1, "AR", "a", "2025-06-22", 200),
    (1, "AR", "d", "2025-08-22", 500),
    (1, "AR", "a", "2025-09-22", 300)
]
source_columns = ["cust_id", "cust_name", "DQ_grade", "date", "balance"]
source_df = spark.createDataFrame(source_data, source_columns).withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# Assume target_df is the SCD2 table (empty for first run)
target_df = spark.createDataFrame([], source_df.schema.add("start_date", DateType()).add("end_date", DateType()).add("is_current", StringType()))

# Prepare source for SCD2
window_spec = Window.partitionBy("cust_id", "cust_name").orderBy("date")
scd2_df = source_df.withColumn("start_date", col("date")) \
    .withColumn("end_date", lead("date").over(window_spec)) \
    .withColumn("is_current", when(lead("date").over(window_spec).isNull(), lit("Y")).otherwise(lit("N"))) \
    .drop("date")

# For the last record, set end_date to a high value (e.g., '9999-12-31')
scd2_df = scd2_df.withColumn("end_date", when(col("end_date").isNull(), lit("9999-12-31").cast(DateType())).otherwise(col("end_date")))

display(scd2_df)